# TransH Link Prediction (FB15k-237)

TransH 논문 (*"Knowledge Graph Embedding by Translating on Hyperplanes"*, AAAI 2014)의
link prediction 실험을 재현하는 코드

- 각 relation마다 초평면(법선벡터 `w_r`)과 그 위에서의 translation `d_r`을 학습
- entity를 관계별 초평면에 투영(projection)한 뒤 TransE와 동일한 방식으로 스코어링
- soft constraint(엔티티 노름 제한, `w_r`-`d_r` 직교 제약)를 loss에 정규화 항으로 추가
- 평가: Mean Rank(MR), MRR, Hits@10 (raw / filtered)

In [ ]:
# 1. 환경 설정
!pip install -q torch --upgrade
import torch
print(torch.__version__, "CUDA:", torch.cuda.is_available())
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
 # 2. 데이터셋 다운로드 (FB15k-237)
import os, urllib.request

DATA_DIR = "FB15k-237"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://github.com/DeepGraphLearning/KnowledgeGraphEmbedding/raw/master/data/FB15k-237"
for fname in ["train.txt", "valid.txt", "test.txt"]:
    url = f"{BASE_URL}/{fname}"
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}", path)
    print(fname, f"downloaded:{os.path.exists(path)}", "size:", sum(1 for _ in open(path, encoding='utf-8')), "lines")

In [ ]:
# 3. 데이터셋 로딩 및 인덱싱
def load_triples(file_path):
    triples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            h, r, t = line.strip().split('\t')
            triples.append((h, r, t))
    return triples

train_raw = load_triples(os.path.join(DATA_DIR, "train.txt"))
valid_raw = load_triples(os.path.join(DATA_DIR, "valid.txt"))
test_raw = load_triples(os.path.join(DATA_DIR, "test.txt"))

entities, relations = set(), set()
for h, r, t in train_raw + valid_raw + test_raw:
    entities.add(h)
    entities.add(t)
    relations.add(r)

ent2id = {e: i for i, e in enumerate(sorted(entities))}
rel2id = {r: i for i, r in enumerate(sorted(relations))}

n_ent, n_rel = len(ent2id), len(rel2id)
print(f"Number of entities: {n_ent}, Number of relations: {n_rel}")

def to_id(triples):
    return torch.tensor([[ent2id[h], rel2id[r], ent2id[t]] for  h , r, t in triples], dtype=torch.long)

train = to_id(train_raw)
valid = to_id(valid_raw)
test = to_id(test_raw)
all_triples_set = set(map(tuple, torch.cat([train, valid, test], dim=0).tolist()))
print(f"Train: {train.shape}, Valid: {valid.shape}, Test: {test.shape}, All triples: {len(all_triples_set)}")





In [ ]:
# 4. TransH 모델 구현
import torch.nn as nn
import torch.nn.functional as F

class TransH(nn.Module):
    def __init__(self, n_ent, n_rel, dim=100, eps=0.001, C=1.0, margin=1.0, norm=2):
        super().__init__()
        self.dim, self.eps, self.C, self.margin, self.norm = dim, eps, C, margin, norm

        self.ent_emb = nn.Embedding(n_ent, dim)
        self.norm_vec = nn.Embedding(n_rel, dim)    # w_r : hyperplane normal vector
        self.trans_vec = nn.Embedding(n_rel, dim)   # d_r : translation vector on the hyperplane
        
        # weight initialization with Xavier uniform distribution
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.norm_vec.weight)
        nn.init.xavier_uniform_(self.trans_vec.weight)

        # w_r normalization
        with torch.no_grad():
            self.norm_vec.weight.data = F.normalize(self.norm_vec.weight.data, p=2, dim=1)
    
    def _project(self, e, w_r):
        # e_perp = e - (w_r^Te) * w_r
        w_r = F.normalize(w_r, p=2, dim=1)
        return e - (e*w_r).sum(dim=1, keepdim=True) * w_r
    
    def score(self, h_idx, r_idx, t_idx):
        h = self.ent_emb(h_idx)
        t = self.ent_emb(t_idx)
        w_r = self.norm_vex(r_idx)
        d_r = self.trans_vec(r_idx)

        h_perp = self._project(h, w_r)
        t_perp = self._project(t, w_r)
        diff = h_perp + d_r - t_perp
        if self.norm == 1:
            return diff.abs().sum(dim=-1)
        return (diff ** 2).sum(dim=-1)
    
    def soft_constraints(self, ent_idx, r_idx):
        # (1) entity norm constraint: ||e||_2 <= 1
        e = self.ent_emb(ent_idx)
        c1 = F.relu(e.norm(p=2, dim=1) - 1.0).mean()

        # (2) orthogonality constraint: |w_r^Td_r| / ||d_r||_2 <= eps
        w_r = F.normalize(self.norm_vec(r_idx), p=2, dim=-1)
        d_r = self.trans_vec(r_idx)
        dot2 = (w_r * d_r).sum(dim=-1) ** 2
        d_norm2 = d_r.norm(p=2, dim=-1) ** 2 + 1e-12 # avoid division by zero
        c2 = F.relu(dot2 / d_norm2 - self.eps ** 2).mean()
        return self.C * (c1 + c2)
    
    def forward(self, pos, neg):
        pos_h, pos_r, pos_t = pos[:, 0], pos[:, 1], pos[:, 2]
        neg_h, neg_r, neg_t = neg[:, 0], neg[:, 1], neg[:, 2]

        pos_score = self.score(pos_h, pos_r, pos_t)
        neg_score = self.score(neg_h, neg_r, neg_t)

        # margin ranking loss : max(0, margin + pos_score - neg_score)
        loss = F.relu(self.margin + pos_score - neg_score).sum()
        loss = loss + self.soft_constraints(torch.cat([pos_h, pos_t]), torch.cat([pos_r, neg_r]))
        return loss

In [ ]:
# 5. Negative Sampling (haed or tail을 랜덤하게 선택하여 교체)
import random

def sample_negatives(batch, n_ent, filter_set):
    neg = batch.clone()
    corrupt_head = torch.rand(len(batch)) < 0.5
    rand_ent = torch.randint(0, n_ent, (len(batch),))

    for i in range(len(batch)):
        h, r, t = batch[i].tolist()
        new_e = rand_ent[i].item()
        tries = 0
        while True:
            cand = (new_e, r, t) if corrupt_head[i] else (h, r, new_e)
            if cand not in filter_set or tries > 10:
                break
            new_e = random.randint(0, n_ent -1)
            tries += 1
        if corrupt_head[i]:
            neg[i, 0] = new_e
        else:
            neg[i, 2] = new_e
    return neg

In [ ]:
# 6. Training
import time
from torch.utils.data import DataLoader, TensorDataset

DIM = 100
BATCH_SIZE = 1024
EPOCHS = 20
LR = 0.01
MARGIN = 1.0
C = 0.25

model = TransH(n_ent, n_rel, dim=DIM, C=C, margin=MARGIN).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

loader = DataLoader(TensorDataset(train), batch_size=BATCH_SIZE, shuffle=True)

loss_history = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    total_loss = 0.0
    # 매 epoch마다 엔티티 임베딩 정규화 (norm_vec은 score 계산 시 정규화되므로 여기서는 제외)
    for (batch,) in loader:
        batch = batch.to(device)
        neg_batch = sample_negatives(batch.cpu(), n_ent, all_triples_set).to(device)

        optimizer.zero_grad()
        loss = model(batch, neg_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    loss_history.append(total_loss)
    if epoch % 2 == 0 :
        print(f"Epoch {epoch}/{EPOCHS}, Loss: {total_loss:12.2f} | Time: {(time.time() - t0):.1f}s")



In [ ]:
# 7. Training loss 시각화
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("TransH Training Loss")
plt.show()

In [ ]:
# 8. Evaluation (Mean Rank / MRR /  Hits@10, raw + filtered)
@torch.no_grad()
def evaluate(model, test_triples, n_ent, filter_set, batch_entities=512, max_test=2000, corrupt='both'):
    model.eval()
    device_ = next(model.parameters()).device
    all_ent_ids = torch.arange(n_ent, device=device_)

    # 평가 속도를 위해 test triples 일부만 샘플링 (전체 평가 시, max_test=None)
    idx = list(range(len(test_triples)))
    if max_test is not None and len(idx) > max_test:
        idx = random.sample(idx, max_test)

    ranks_raw, ranks_filt = [], []

    for i in idx:
        h, r, t = test_triples[i].tolist()
        r_t = torch.tensor(r, device=device_)

        targets = ['tail', 'head'] if corrupt == 'both' else [corrupt]
        for target in targets:
            if target == 'tail':
                h_batch = torch.full((n_ent,), h, dtype=torch.long, device=device_)
                r_batch = torch.full((n_ent,), r, dtype=torch.long, device=device_)
                t_batch = all_ent_ids
                correc_id = t
            else:  # target == 'head'
                h_batch = all_ent_ids
                r_batch = torch.full((n_ent,), r, dtype=torch.long, device=device_)
                t_batch = torch.full((n_ent,), t, dtype=torch.long, device=device_)
                correct_id = h

            scores = model.score(h_batch, r_batch, t_batch)
            order = torch.argsort(scores)
            rank_raw = (order == correct_id).nonzero(as_tuple=True)[0].item() + 1
            ranks_raw.append(rank_raw)

            # filtered rank 계산
            sorted_ids = order.tolist()
            filt_rank = 0
            for cand in sorted_ids:
                if cand == correct_id:
                    filt_rank += 1
                    break
                cand_triple = (cand, r, t) if target == 'tail' else (h, r, cand)
                if cand_triple in filter_set:
                    continue
                filt_rank += 1
            ranks_filt.append(filt_rank)
    
    ranks_raw = torch.tensor(ranks_raw, dtype=torch.float)
    ranks_filt = torch.tensor(ranks_filt, dtype=torch.float)

    def stats(ranks):
        mean_rank = ranks.mean().item()
        mrr = (1.0 / ranks).mean().item()
        hits10 = (ranks <= 10).float().mean().item()
        return mean_rank, mrr, hits10
    
    mean_rank_raw, mrr_raw, hits10_raw = stats(ranks_raw)
    mean_rank_filt, mrr_filt, hits10_filt = stats(ranks_filt)

    print(f"[Raw] Mean Rank: {mean_rank_raw:.2f}, MRR: {mrr_raw:.4f}, Hits@10: {hits10_raw:.4f}")
    print(f"[Filtered] Mean Rank: {mean_rank_filt:.2f}, MRR: {mrr_filt:.4f}, Hits@10: {hits10_filt:.4f}")
    return dict(mean_rank_raw=mean_rank_raw, mrr_raw=mrr_raw, hits10_raw=hits10_raw,
                mean_rank_filt=mean_rank_filt, mrr_filt=mrr_filt, hits10_filt=hits10_filt)

results = evaluate(model, test, n_ent, all_triples_set, max_test=2000, corrupt='both')


## 참고 / 튜닝 포인트

- `EPOCHS`, `DIM`, `LR`, `MARGIN`, `C`(soft constraint 가중치)는 TransE / TransH 논문 참고
- 평가(`evaluate`)에서 `max_test`를 늘리면(예: `None`으로 전체 test set) 논문과 동일한 전체 평가가 되지만,
  entity 수 × test triple 수만큼 forward pass가 필요해 CPU에서는 매우 느림. GPU 런타임 사용 권장
  (`런타임 > 런타임 유형 변경 > GPU`)
- negative sampling은 현재 uniform random 사용. 논문의 "bern" (relation의 하나의 head/tail 의 평균 tail/head 개수에 따라 확률적으로 head/tail 교체) 방식으로 바꾸면 성능이 더 오를 수도 있음.